# QML Autoencoder - Complete Mac Training 🍎

**Features:**
- ✅ All fixes applied (device + numpy)
- ✅ Multiple epochs with progress tracking
- ✅ Final results summary
- ✅ Model saving
- ✅ Loss visualization

**Run all cells to complete full training!**

In [1]:
import numpy as np
import torch
import torch.nn as nn
import pennylane as qml
import matplotlib.pyplot as plt
from datetime import datetime

print("="*60)
print("QML AUTOENCODER - COMPLETE MAC TRAINING")
print("="*60)
print(f"PyTorch: {torch.__version__} | PennyLane: {qml.__version__}")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

QML AUTOENCODER - COMPLETE MAC TRAINING
PyTorch: 2.8.0 | PennyLane: 0.42.3
Start time: 2025-10-30 16:23:44


In [2]:
# Configuration
n_qubits = 4
n_layers = 2
latent_dim = 16
img_size = 128
batch_size = 8
NUM_WORKERS = 0
n_epochs = 5  # Number of epochs to train

print(f"\n{'='*60}")
print("TRAINING CONFIGURATION:")
print(f"{'='*60}")
print(f"  Qubits: {n_qubits}")
print(f"  Quantum layers: {n_layers}")
print(f"  Latent dimension: {latent_dim}")
print(f"  Image size: {img_size}x{img_size}")
print(f"  Batch size: {batch_size}")
print(f"  Epochs: {n_epochs}")
print(f"  Estimated time: {n_epochs * 7} minutes")
print(f"{'='*60}\n")


TRAINING CONFIGURATION:
  Qubits: 4
  Quantum layers: 2
  Latent dimension: 16
  Image size: 128x128
  Batch size: 8
  Epochs: 5
  Estimated time: 35 minutes



In [3]:
from torch.utils.data import Subset, DataLoader

print("Loading data...\n")

try:
    from preprocess import train_dataset, test_dataset
    print("✅ Found preprocess.py - using ChestMNIST data!")
    
    def get_single_label_indices(dataset):
        indices = []
        for i in range(0, len(dataset), 1000):
            end = min(i + 1000, len(dataset))
            labels = []
            for j in range(i, end):
                _, label = dataset[j]
                if isinstance(label, np.ndarray):
                    label = torch.from_numpy(label)
                labels.append(label)
            labels = torch.stack(labels)
            mask = (labels.sum(dim=1) == 1)
            indices.extend(torch.arange(i, end)[mask].tolist())
        return indices
    
    print("Filtering for single-label images...")
    train_idx = get_single_label_indices(train_dataset)
    test_idx = get_single_label_indices(test_dataset)
    train_dataset = Subset(train_dataset, train_idx)
    test_dataset = Subset(test_dataset, test_idx)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    print(f"✅ Training: {len(train_dataset)} samples ({len(train_loader)} batches)")
    print(f"✅ Test: {len(test_dataset)} samples ({len(test_loader)} batches)\n")
except:
    print("⚠️ preprocess.py not found - using dummy data\n")
    from torch.utils.data import TensorDataset
    imgs = torch.randn(80, 1, img_size, img_size)
    labels = torch.zeros(80, 14)
    for i in range(80): labels[i, i % 14] = 1
    train_dataset = TensorDataset(imgs[:64], labels[:64])
    test_dataset = TensorDataset(imgs[64:], labels[64:])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    print(f"✅ Dummy data: {len(train_dataset)} train, {len(test_dataset)} test\n")

Loading data...

✅ Found preprocess.py - using ChestMNIST data!
Filtering for single-label images...
✅ Training: 21602 samples (2700 batches)
✅ Test: 6259 samples (783 batches)



In [4]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=16, img_size=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, latent_dim))
    def forward(self, x): return self.encoder(x)

print("✅ Encoder defined")

✅ Encoder defined


In [5]:
try:
    dev = qml.device("lightning.qubit", wires=n_qubits)
    print("✅ Using lightning.qubit (fast CPU simulator)")
except:
    dev = qml.device("default.qubit", wires=n_qubits)
    print("Using default.qubit")

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def qnode_single(inputs, weights):
    qml.AmplitudeEmbedding(inputs, wires=range(n_qubits), normalize=True, pad_with=0.0)
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

print("✅ Quantum circuit defined")

✅ Using lightning.qubit (fast CPU simulator)
✅ Quantum circuit defined


In [6]:
class QuantumHeadOptimized(nn.Module):
    def __init__(self, n_layers, n_qubits, n_classes, latent_dim):
        super().__init__()
        self.n_qubits = n_qubits
        self.encoder_fc = nn.Linear(latent_dim, 2**n_qubits)
        nn.init.xavier_uniform_(self.encoder_fc.weight, gain=0.01)
        nn.init.zeros_(self.encoder_fc.bias)
        self.q_weights = nn.Parameter(torch.randn(n_layers, n_qubits, 3) * 0.001)
        self.readout = nn.Linear(n_qubits, n_classes)
    
    def _prepare_quantum_input(self, h):
        z = self.encoder_fc(h)
        z = torch.nan_to_num(torch.clamp(z, -5, 5))
        norm = torch.sqrt(torch.sum(z**2, dim=1, keepdim=True) + 1e-10)
        if (norm.squeeze() < 1e-6).any():
            uniform = torch.ones(2**self.n_qubits, device=z.device) / np.sqrt(2**self.n_qubits)
            z[norm.squeeze() < 1e-6] = uniform
            norm = torch.sqrt(torch.sum(z**2, dim=1, keepdim=True) + 1e-10)
        return z / norm
    
    def forward(self, h):
        device = h.device
        z_norm = self._prepare_quantum_input(h).cpu()
        q_w_cpu = self.q_weights.cpu()
        results = []
        for i in range(h.shape[0]):
            try:
                expvals = qnode_single(z_norm[i].detach(), q_w_cpu)
                results.append(torch.stack(expvals).float())
            except:
                results.append(torch.zeros(self.n_qubits))
        return self.readout(torch.stack(results).to(device))

print("✅ Quantum head defined")

✅ Quantum head defined


In [7]:
class HybridQML(nn.Module):
    def __init__(self, img_size, latent_dim, n_classes=14):
        super().__init__()
        self.enc = Encoder(latent_dim, img_size)
        self.qhead = QuantumHeadOptimized(n_layers, n_qubits, n_classes, latent_dim)
    def forward(self, x, return_recon=False):
        return self.qhead(self.enc(x))

print("✅ Hybrid model defined")

✅ Hybrid model defined


In [8]:
device = torch.device("cpu")
print(f"\n🚀 Device: {device}\n")

model = HybridQML(img_size, latent_dim, 14).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-5, eps=1e-7)
criterion = nn.BCEWithLogitsLoss()

print(f"✅ Model initialized: {sum(p.numel() for p in model.parameters()):,} parameters\n")


🚀 Device: cpu

✅ Model initialized: 95,102 parameters



In [9]:
import time

def train_one_epoch(epoch):
    model.train()
    total, n = 0.0, 0
    start_time = time.time()
    
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{n_epochs}")
    print(f"{'='*60}")
    
    for i, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(device), labels.float().to(device)
        logits = model(imgs, True)
        loss = criterion(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * imgs.size(0)
        n += imgs.size(0)
        
        if i % max(1, len(train_loader) // 10) == 0:
            elapsed = time.time() - start_time
            progress = (i + 1) / len(train_loader) * 100
            print(f"  [{progress:5.1f}%] Batch {i:3d}/{len(train_loader)} | Loss: {total/n:.4f} | Time: {elapsed:.1f}s")
    
    epoch_time = time.time() - start_time
    avg_loss = total / n
    print(f"\n  ✅ Epoch {epoch} complete in {epoch_time:.1f}s | Train Loss: {avg_loss:.4f}")
    return avg_loss, epoch_time

@torch.no_grad()
def evaluate():
    model.eval()
    total, n = 0.0, 0
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.float().to(device)
        loss = criterion(model(imgs), labels)
        total += loss.item() * imgs.size(0)
        n += imgs.size(0)
    return total / n

print("✅ Training functions defined")

✅ Training functions defined


In [ ]:
print("\n" + "="*60)
print("STARTING COMPLETE TRAINING")
print("="*60)
print(f"Training for {n_epochs} epochs...\n")

# Track results
train_losses = []
test_losses = []
epoch_times = []
best_test_loss = float('inf')
best_epoch = 0

total_start = time.time()

# Training loop
for epoch in range(1, n_epochs + 1):
    train_loss, epoch_time = train_one_epoch(epoch)
    test_loss = evaluate()
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    epoch_times.append(epoch_time)
    
    # Track best model
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_epoch = epoch
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'test_loss': test_loss,
        }, 'best_model.pt')
        print(f"  💾 New best model saved! (Test Loss: {test_loss:.4f})")
    
    print(f"  📊 Test Loss: {test_loss:.4f} | Best: {best_test_loss:.4f} (Epoch {best_epoch})")

total_time = time.time() - total_start

print("\n" + "="*60)
print("TRAINING COMPLETE! 🎉")
print("="*60)


STARTING COMPLETE TRAINING
Training for 5 epochs...


EPOCH 1/5
  [  0.0%] Batch   0/2700 | Loss: 0.7156 | Time: 0.4s
  [ 10.0%] Batch 270/2700 | Loss: 0.6926 | Time: 34.9s
  [ 20.0%] Batch 540/2700 | Loss: 0.6866 | Time: 69.2s
  [ 30.0%] Batch 810/2700 | Loss: 0.6807 | Time: 103.3s
  [ 40.0%] Batch 1080/2700 | Loss: 0.6750 | Time: 137.4s
  [ 50.0%] Batch 1350/2700 | Loss: 0.6694 | Time: 172.4s
  [ 60.0%] Batch 1620/2700 | Loss: 0.6639 | Time: 207.2s
  [ 70.0%] Batch 1890/2700 | Loss: 0.6583 | Time: 242.4s


In [ ]:
# Final Results Summary
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(f"\n📊 Training Statistics:")
print(f"  Total epochs: {n_epochs}")
print(f"  Total time: {total_time/60:.1f} minutes ({total_time:.1f} seconds)")
print(f"  Average time per epoch: {np.mean(epoch_times):.1f} seconds")
print(f"\n📈 Loss Progression:")
print(f"  Initial train loss: {train_losses[0]:.4f}")
print(f"  Final train loss: {train_losses[-1]:.4f}")
print(f"  Improvement: {(train_losses[0] - train_losses[-1])/train_losses[0]*100:.1f}%")
print(f"\n  Initial test loss: {test_losses[0]:.4f}")
print(f"  Final test loss: {test_losses[-1]:.4f}")
print(f"  Best test loss: {best_test_loss:.4f} (Epoch {best_epoch})")
print(f"  Improvement: {(test_losses[0] - best_test_loss)/test_losses[0]*100:.1f}%")
print(f"\n💾 Model saved as: best_model.pt")
print(f"\n⏱️  Performance:")
print(f"  Mac training time: {total_time/60:.1f} minutes")
print(f"  Estimated Scholar time: ~{n_epochs * 0.5:.1f} minutes (120x faster!)")
print("\n" + "="*60)

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 4))

# Loss plot
plt.subplot(1, 2, 1)
epochs_range = range(1, n_epochs + 1)
plt.plot(epochs_range, train_losses, 'b-o', label='Train Loss', linewidth=2, markersize=8)
plt.plot(epochs_range, test_losses, 'r-s', label='Test Loss', linewidth=2, markersize=8)
plt.axhline(y=best_test_loss, color='g', linestyle='--', label=f'Best Test ({best_test_loss:.4f})', linewidth=1)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Progress', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Time plot
plt.subplot(1, 2, 2)
plt.bar(epochs_range, epoch_times, color='steelblue', alpha=0.7)
plt.axhline(y=np.mean(epoch_times), color='r', linestyle='--', label=f'Average ({np.mean(epoch_times):.1f}s)', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Time (seconds)', fontsize=12)
plt.title('Time per Epoch', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
print("\n📊 Training curves saved as: training_results.png")
plt.show()

In [ ]:
# Save final model and training history
torch.save({
    'n_epochs': n_epochs,
    'final_epoch': n_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'test_losses': test_losses,
    'epoch_times': epoch_times,
    'best_test_loss': best_test_loss,
    'best_epoch': best_epoch,
    'config': {
        'n_qubits': n_qubits,
        'n_layers': n_layers,
        'latent_dim': latent_dim,
        'img_size': img_size,
        'batch_size': batch_size,
    }
}, 'final_model_complete.pt')

print("\n💾 Complete training history saved as: final_model_complete.pt")
print("\nThis includes:")
print("  - Final model weights")
print("  - All loss curves")
print("  - Training times")
print("  - Configuration")
print("\n" + "="*60)
print("ALL DONE! Ready to upload to Scholar for 120x speedup! 🚀")
print("="*60)